[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baluragala/building-rag-pipelines/blob/main/notebooks/08_conclusion_end_to_end.ipynb)

# Building RAG Pipelines
## Notebook 08: Conclusion — The Full Pipeline & Debugging
**Duration:** 10 min &nbsp;|&nbsp; **Mode:** End-to-end

> Taught **WHY → WHAT → HOW**. We keep asking *"What happens if this step is poorly
> designed?"* and we **predict before we run** and **compare outputs**. LangChain is
> shown as a **parallel mapping** — it abstracts mechanics but not design decisions.

![pipeline](https://dummyimage.com/1000x70/1f2937/ffffff&text=Loading+%E2%86%92+Chunking+%E2%86%92+Retrieval+%E2%86%92+Augmentation+%E2%86%92+Generation+%E2%86%92+Evaluation)

In [ ]:
# ============================================================
# COLAB BOOTSTRAP — run this cell first. (Same as every notebook.)
# ============================================================
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/baluragala/building-rag-pipelines.git"  # INSTRUCTOR: set this

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

_pip("numpy", "openai", "tiktoken", "rank-bm25", "beautifulsoup4", "pypdf",
     "langchain-community", "langchain-text-splitters", "langchain-openai", "faiss-cpu")
try:
    import rag_pipeline
except ModuleNotFoundError:
    if IN_COLAB:
        subprocess.run(["git", "clone", "-q", REPO_URL], check=False)
        if os.path.isdir("building-rag-pipelines"):
            sys.path.insert(0, "building-rag-pipelines")
        else:
            print("Clone failed. Upload `rag_pipeline/` + `data/` via the Colab file browser, then re-run.")
    else:
        sys.path.insert(0, os.path.abspath(".."))
    import rag_pipeline

def data_path(*parts):
    for base in ("data", "../data", "building-rag-pipelines/data"):
        p = os.path.join(base, *parts)
        if os.path.exists(p):
            return p
    return os.path.join("data", *parts)

print("rag_pipeline", rag_pipeline.__version__, "ready.  Colab:", IN_COLAB)

In [ ]:
# ============================================================
# CHOOSE YOUR PROVIDERS  (OpenAI is the default)
# ============================================================
# Default stack = OpenAI: gpt-4o-mini (LLM) + text-embedding-3-small (embeddings).
# In Colab the key is read automatically from the Colab SECRETS manager:
#   left sidebar -> key icon -> add a secret named OPENAI_API_KEY
#   -> toggle "Notebook access" ON  -> re-run this cell.
# If no key is found anywhere, we fall back to the offline MOCK so the notebook
# still runs end-to-end.
import os

def _load_openai_key():
    if os.getenv("OPENAI_API_KEY"):
        return True
    try:  # Colab Secrets
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
        if key:
            os.environ["OPENAI_API_KEY"] = key
            return True
    except Exception:
        pass  # not in Colab, secret missing, or access not granted
    return False

if _load_openai_key():
    os.environ.setdefault("RAG_LLM_PROVIDER", "openai")
    os.environ.setdefault("RAG_EMBED_PROVIDER", "openai")
    print("OpenAI key found -> using the OpenAI stack.")
else:
    os.environ["RAG_LLM_PROVIDER"] = "mock"
    os.environ["RAG_EMBED_PROVIDER"] = "mock"
    print("No OPENAI_API_KEY found -> using the offline MOCK providers.\n"
          "In Colab: add a Secret named OPENAI_API_KEY (key icon, left sidebar),\n"
          "enable Notebook access, and re-run this cell to switch to OpenAI.")

from rag_pipeline import config
print(config.current_config())

In [ ]:
from rag_pipeline.loaders import load_directory
docs = load_directory(data_path("corpus"))
print(f"Loaded {len(docs)} documents from the Acme Cloud corpus.")

## Connecting all six stages

Everything we built is one pipeline, and every stage is a **knob**:

```
Loading → Chunking → Retrieval → Augmentation → Generation → Evaluation
 clean+   size/      dense/       stuff/          grounded+    metrics +
 metadata overlap    hybrid/rerank map-reduce/     citations    diagnosis
                                   refine
```

`RAGPipeline` exposes each knob. The session's thesis, made executable: **change a
knob, change the answer — quality is cumulative across all stages.**

## Show how design choices affect output — a mini config sweep

In [ ]:
from rag_pipeline import config, evaluation as ev
from rag_pipeline.pipeline import RAGPipeline
from rag_pipeline.chunking import fixed_size_chunk, recursive_chunk
from rag_pipeline.retrieval import DenseRetriever, BM25Retriever, HybridRetriever

emb, llm = config.get_embedder(), config.get_llm()
examples = ev.load_eval_dataset(data_path("eval", "eval_dataset.jsonl"))

def hybrid_factory(store, embedder):
    # HybridRetriever needs the chunk list; rebuild BM25 from the store's docs.
    dense = DenseRetriever(store, embedder)
    return HybridRetriever(dense, BM25Retriever(store.documents))

configs = {
    "A: tiny fixed chunks, dense, k=1": dict(
        chunker=lambda d: fixed_size_chunk(d, 120, 0), k=1),
    "B: recursive chunks, dense, k=3": dict(
        chunker=lambda d: recursive_chunk(d, 500, 50), k=3),
    "C: recursive chunks, HYBRID, k=3": dict(
        chunker=lambda d: recursive_chunk(d, 500, 50),
        retriever_factory=hybrid_factory, k=3),
}
for name, cfg in configs.items():
    pipe = RAGPipeline(embedder=emb, llm=llm, **cfg).ingest(docs)
    k = cfg["k"]
    scores = ev.evaluate_retriever(pipe.retriever, examples, k=k)
    print(f"{name:<40} recall@{k}={scores['recall@' + str(k)]:.3f}  mrr={scores['mrr']:.3f}")

**What you should observe:** config A (tiny chunks, dense, k=1) scores worst; each
*design decision* toward B and C lifts retrieval quality — with the **same LLM**.
That is the whole session in one table: *it isn't the model, it's the pipeline.*

## Walk a complete example — and debug a BAD output

Take config A's bad answer to a multi-hop question and diagnose it stage by stage
using the trace, exactly as you would in production.

In [ ]:
q = "Which plan gives SSO and a 99.99% SLA?"

bad = RAGPipeline(embedder=emb, llm=llm,
                  chunker=lambda d: fixed_size_chunk(d, 120, 0), k=1).ingest(docs)
bad_out = bad.query(q)
print("BAD ANSWER:", bad_out["answer"][:160])
print("Retrieved chunks:", len(bad_out["sources"]), "-> too few for a multi-doc question")
print()
print("DEBUG WORKFLOW:")
print(" 1) Retrieval first: only 1 tiny chunk retrieved -> RETRIEVAL failure, not the LLM.")
print(" 2) Fix the failing stage: bigger chunks + hybrid + higher k.")

In [ ]:
# Apply the fix and re-run the SAME question.
good = RAGPipeline(embedder=emb, llm=llm,
                   chunker=lambda d: recursive_chunk(d, 500, 50),
                   retriever_factory=hybrid_factory, k=4).ingest(docs)
good_out = good.query(q)
print("FIXED ANSWER:", good_out["answer"][:220])
print("Sources:", [s["label"] for s in good_out["sources"]])

## The debugging workflow (keep this checklist)

When a RAG answer is wrong, go **upstream to downstream**:
1. **Retrieval** — did the right chunk get retrieved at all? (check `trace`, recall@k)
   - No → fix **loading / chunking / retriever / k / hybrid / rerank**.
2. **Augmentation** — was the retrieved chunk actually in the context? (budget/order)
3. **Generation** — right context but wrong answer? → **faithfulness** low → fix the
   **prompt / grounding**.
Never start by blaming the LLM. It's usually an upstream stage.

## The one takeaway

> **RAG performance is determined by cumulative design decisions across all stages
> — loading, chunking, retrieval, augmentation, generation, evaluation — not just
> the choice of LLM.**

You can now **design, implement, and evaluate** a RAG system by making informed
choices at each stage, and **diagnose** failures by localising the responsible
stage. Frameworks help you move faster, but the design decisions remain yours.

### Additional reading
- OpenAI: Retrieval & Vector embeddings docs
- `sentence-transformers` documentation
- LangChain: Retrieval docs
- RAGAS documentation